# NASA C-MAPSS Dataset - Data Loading

## Goal

The goal of this notebook is to load the raw NASA C-MAPSS FD001 training dataset and understand its basic structure.

We inspect:

- dataset shape
- column names
- first rows
- number of engines
- cycle distribution

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_cmapss_data

In [2]:
train_path = PROJECT_ROOT / "data" / "raw" / "train_FD001.txt"

df = load_cmapss_data(train_path)

df.head()

,id,cycle,setting_1,setting_2,setting_3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [3]:
print("Dataset shape:", df.shape)
print("Number of engines:", df["id"].nunique())
print("Number of columns:", len(df.columns))
print("Minimum cycle:", df["cycle"].min())
print("Maximum cycle:", df["cycle"].max())

assert df.shape == (20631, 26)
assert df["id"].nunique() == 100
assert len(df.columns) == 26

Dataset shape: (20631, 26)
Number of engines: 100
Number of columns: 26
Minimum cycle: 1
Maximum cycle: 362


## Why use a shared data loader?
Instead of duplicating the dataset loading code in every notebook, the project uses a shared utility function from src/data_loader.py. This follows the DRY (Don’t Repeat Yourself) principle, improves maintainability, and ensures that any future changes to the loading process only need to be made in one place.

## Column Overview

The dataset contains:

- `id`: engine identifier
- `cycle`: time step for each engine
- `setting_1` to `setting_3`: operational settings
- `s1` to `s21`: sensor measurements

In [4]:
df.columns.tolist()

['id',
 'cycle',
 'setting_1',
 'setting_2',
 'setting_3',
 's1',
 's2',
 's3',
 's4',
 's5',
 's6',
 's7',
 's8',
 's9',
 's10',
 's11',
 's12',
 's13',
 's14',
 's15',
 's16',
 's17',
 's18',
 's19',
 's20',
 's21']

## Insight

Each row represents one engine at one cycle.

The `id` column identifies the engine, while `cycle` represents the time progression for that engine.

The dataset contains operational settings and 21 sensor measurements, which will later be used to analyze engine degradation and predict Remaining Useful Life (RUL).

## Data validation

Before continuing with exploratory analysis and feature engineering, the dataset is checked for missing values, duplicated engine-cycle records, invalid cycle numbers, and discontinuities in engine histories.

These checks help detect data quality issues early and prevent silent errors from propagating through the pipeline.

In [5]:
missing_values = df.isna().sum()

missing_values[missing_values > 0]

Series([], dtype: int64)

In [6]:
assert df.isna().sum().sum() == 0, "Dataset contains missing values."

In [7]:
duplicate_engine_cycles = df.duplicated(
    subset=["id", "cycle"]
).sum()

print("Duplicated engine-cycle records:", duplicate_engine_cycles)

Duplicated engine-cycle records: 0


In [8]:
assert duplicate_engine_cycles == 0, (
    "Duplicate engine-cycle records were found."
)

In [9]:
invalid_cycles = df[df["cycle"] < 1]

print("Rows with invalid cycle values:", len(invalid_cycles))

Rows with invalid cycle values: 0


In [10]:
assert invalid_cycles.empty, (
    "Cycle values must be greater than or equal to 1."
)

In [11]:
first_cycles = df.groupby("id")["cycle"].min()

first_cycles.value_counts().sort_index()

cycle
1    100
Name: count, dtype: int64

In [12]:
assert first_cycles.eq(1).all(), (
    "Some engine histories do not start at cycle 1."
)

In [13]:
cycle_continuity = (
    df.sort_values(["id", "cycle"])
      .groupby("id")["cycle"]
      .apply(lambda cycles: cycles.diff().dropna().eq(1).all())
)

In [14]:
discontinuous_engines = cycle_continuity[
    ~cycle_continuity
].index.tolist()

print("Engines with discontinuous cycles:", discontinuous_engines)

Engines with discontinuous cycles: []


In [15]:
assert cycle_continuity.all(), (
    f"Discontinuous cycle histories found for engines: "
    f"{discontinuous_engines}"
)

In [16]:
print("Data validation passed successfully.")
print(f"Rows: {len(df):,}")
print(f"Engines: {df['id'].nunique()}")
print(f"Columns: {df.shape[1]}")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Duplicate engine-cycle rows: {duplicate_engine_cycles}")

Data validation passed successfully.
Rows: 20,631
Engines: 100
Columns: 26
Missing values: 0
Duplicate engine-cycle rows: 0


### Validation insight

The FD001 training dataset passed all validation checks:

- no missing values were found,
- each engine-cycle pair is unique,
- all engine histories begin at cycle 1,
- cycle values are positive,
- and the cycle sequences are continuous for every engine.

This confirms that the training data is structurally consistent and ready for RUL construction and exploratory analysis.